# 🥇 Notebook 03 — Gold Layer: Build Star Schema Fact Tables

**Goal:** Join Silver dimension tables with Bronze transaction/loan data to build two Gold Fact Tables. This completes the star schema.

> **Run time:** ~5 min

## Star Schema Architecture
```
                    dim_date
                       │
         dim_branch ───┤
                       │
  dim_customer ─── fact_transactions ─── dim_account
                       │
                   dim_product (via loan join)


  dim_customer ─── fact_loans ─── dim_account
                       │              │
                   dim_product    dim_branch
                       │
                   dim_date
```

## Why Star Schema?
- **Queries are simpler** — one fact table + JOIN to dims
- **Power BI Direct Lake** reads star schemas natively — no aggregations needed
- **Performance** — Delta Tables with Z-ordering on join keys
- **Governed** — each dimension is a single source of truth

In [ ]:
from pyspark.sql import functions as F

print('Building Gold star schema fact tables...')

## fact_transactions — 2,000 transaction events

Each row = one transaction event. Foreign keys link to every dimension.

In [ ]:
# Load silver dims
dim_customer = spark.table('dim_customer')
dim_account  = spark.table('dim_account')
dim_branch   = spark.table('dim_branch')
dim_date     = spark.table('dim_date')

# Load bronze transactions (fact source)
bronze_txn = spark.table('bronze_transactions') \
    .drop('_ingested_at','_source_file')

fact_transactions = bronze_txn \
    .withColumn('TransactionDateKey', F.date_format(F.to_date('TransactionDate','yyyy-MM-dd'), 'yyyyMMdd').cast('int')) \
    .join(dim_date.select('DateKey','Year','Quarter','MonthName','IsWeekend'),
          F.col('TransactionDateKey') == F.col('DateKey'), 'left') \
    .drop('DateKey') \
    .join(dim_account.select('AccountID','AccountType','BranchID','Balance','BalanceTier','Status'),
          on='AccountID', how='left') \
    .join(dim_customer.select('CustomerID','FullName','CustomerSegment','CreditScoreTier','City','State'),
          on='CustomerID', how='left') \
    .join(dim_branch.select('BranchID','BranchName','Region'),
          on='BranchID', how='left') \
    .withColumn('IsLargeTransaction', F.col('Amount') >= 10000) \
    .withColumn('AmountBand',
        F.when(F.col('Amount') < 100,   'Micro (<$100)')
         .when(F.col('Amount') < 1000,  'Small ($100-$999)')
         .when(F.col('Amount') < 5000,  'Medium ($1K-$4.9K)')
         .when(F.col('Amount') < 25000, 'Large ($5K-$24.9K)')
         .otherwise('Very Large ($25K+)'))

fact_transactions.write.format('delta').mode('overwrite').saveAsTable('fact_transactions')
print(f'fact_transactions: {fact_transactions.count()} rows, {len(fact_transactions.columns)} columns')
fact_transactions.show(5)

## fact_loans — 500 loan records

Each row = one loan. Joins to all 5 dimensions.

In [ ]:
dim_product = spark.table('dim_product')
bronze_loans = spark.table('bronze_loans').drop('_ingested_at','_source_file')

fact_loans = bronze_loans \
    .withColumn('StartDateKey', F.date_format(F.to_date('StartDate','yyyy-MM-dd'), 'yyyyMMdd').cast('int')) \
    .join(dim_date.select(F.col('DateKey').alias('StartDateKey'),'Year','Quarter','MonthName'),
          on='StartDateKey', how='left') \
    .join(dim_customer.select('CustomerID','FullName','CustomerSegment','CreditScoreTier','City','State','AgeGroup'),
          on='CustomerID', how='left') \
    .join(dim_account.select('AccountID','AccountType','Balance','BalanceTier'),
          on='AccountID', how='left') \
    .join(dim_product.select('ProductID','ProductName','ProductType','MinInterestRate','MaxInterestRate'),
          on='ProductID', how='left') \
    .join(dim_branch.select('BranchID','BranchName','Region'),
          on='BranchID', how='left') \
    .withColumn('IsDefault',      F.col('LoanStatus').isin(['Default','90-Days Late']).cast('int')) \
    .withColumn('IsDelinquent',   F.col('LoanStatus').isin(['30-Days Late','60-Days Late','90-Days Late','Default']).cast('int')) \
    .withColumn('LoanToBalance',  F.round(F.col('LoanAmount') / (F.col('Balance') + F.lit(1)), 4))

fact_loans.write.format('delta').mode('overwrite').saveAsTable('fact_loans')
print(f'fact_loans: {fact_loans.count()} rows, {len(fact_loans.columns)} columns')
fact_loans.show(5)

## Validate Star Schema: JOIN Queries

In [ ]:
%%sql
-- 1. Transaction volume by branch region and quarter
SELECT
    Region,
    QuarterName,
    Year,
    COUNT(*)                AS NumTransactions,
    ROUND(SUM(Amount), 0)   AS TotalAmount,
    ROUND(AVG(Amount), 2)   AS AvgAmount
FROM fact_transactions
WHERE Year IS NOT NULL
GROUP BY Region, QuarterName, Year
ORDER BY Year, QuarterName, TotalAmount DESC
LIMIT 20

In [ ]:
%%sql
-- 2. Loan default rate by customer segment and product type
SELECT
    CustomerSegment,
    ProductType,
    COUNT(*)                                        AS TotalLoans,
    SUM(LoanAmount)                                 AS TotalLoanValue,
    SUM(IsDefault)                                  AS Defaults,
    ROUND(SUM(IsDefault) * 100.0 / COUNT(*), 1)     AS DefaultRate_Pct,
    ROUND(AVG(InterestRate), 2)                     AS AvgInterestRate
FROM fact_loans
GROUP BY CustomerSegment, ProductType
ORDER BY DefaultRate_Pct DESC

In [ ]:
%%sql
-- 3. Customer 360: join transactions + loans per customer
SELECT
    c.CustomerID,
    c.FullName,
    c.CreditScoreTier,
    c.CustomerSegment,
    COUNT(DISTINCT t.TransactionID)  AS NumTransactions,
    ROUND(SUM(t.Amount), 2)          AS TotalTxnAmount,
    COUNT(DISTINCT l.LoanID)         AS NumLoans,
    ROUND(SUM(l.LoanAmount), 2)      AS TotalLoanAmount,
    SUM(l.IsDefault)                 AS LoanDefaults
FROM dim_customer c
LEFT JOIN fact_transactions t ON c.CustomerID = t.CustomerID
LEFT JOIN fact_loans        l ON c.CustomerID = l.CustomerID
GROUP BY c.CustomerID, c.FullName, c.CreditScoreTier, c.CustomerSegment
ORDER BY TotalLoanAmount DESC
LIMIT 15